In [1]:
import numpy as np, pickle, os, time as time_lib, sys, csv
import matplotlib.pyplot as plt

In [2]:
sys.path.append('/home/filip/Git_Code/')

from demler_tools.file_manager import path_management, file_management, io
from demler_tools.file_manager import path_management, file_management_local_backend
from demler_tools.file_manager import request_creation                                                                               
from demler_tools.file_manager import file_management,cluster_backend_tools
from demler_tools.parallelization import local_run_parallel, cluster_interface

path_management.initialize(project_name = 'psBQP-keldysh')

Successfully initialized path management with following parameters:
     username:                fmarijanovic
     project name:            psBQP-keldysh
     controlling machine:     laptop
     library version:         v2.3.0-beta-17-g6b0081d
     auto SSH transfer:       True
     default save location:   /home/filip/Documents/Research/data_files/
     LTS mount point:         /home/filip
     email domain:            phys.ethz.ch
     cluster OS:              Ubuntu
     cluster partition:       work
     SSH host file:           /home/filip/.ssh/known_hosts
     SSH key file:            /home/filip/.ssh/id_ed25519_euler
     default modules:         {'CentOS': 'gcc/8.2.0 python/3.11.2', 'Ubuntu': 'stack/2024-06 python/3.11.6'}


In [3]:
crt_controlling_machine = 'laptop'
crt_simulations_machine = 'cluster_euler'

In [4]:
print(np.__version__)
print(path_management.CRT_PATH_CONVENTION)

1.24.2
v3


## Cluster run

In [5]:
crt_simulations_machine = 'cluster_euler'


### Making a job request -- looping vector potential values

In [ ]:
simulation_type = 'Occupation tracking test'
simulation_desc = 'Testing occupation tracking'


_loop_kwargs = 'field_params'
evolution_timesteps = 10
initial_state_timestamp = '1782219990' #1782196253 -- eta = 0.2; 1782219875 -- eta = 0.1
initial_state_index = 8
grid_parameters = {'time_sampling': 1501, 'time_duration': 2 * np.pi * 5}
#vector_potential = np.ones(evolution_timesteps) * 0.0
#temp_list = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8]
system_parameters = {'critical_temperature': 1, 'temperature': 0.5, 'eta': 0.2}
#for temp in temp_list:
#    system_parameters += []

field_type = 'constant'
occupation_tracking_every_n = 1 

save_full_state = False

field_params = []

vec_potential_mags = [0.1,0.5]

#frequencies = [0.1,0.2,0.5,1.0,2.0]
#for fwhm in FWHMS: 
FWHMs = np.linspace(1.0,5.0,5,endpoint=True)
#FWHMs = np.linspace(0.1,5.0,15,endpoint=True)
FWHMs = [0.0]
amplitudes = np.linspace(0.05,1.0,10,endpoint=True)
amplitudes = [0.00]
for fwhm in FWHMs:
    for amplitude in amplitudes:
        field_params += [{'amplitude': amplitude, 'FWHM': fwhm, 'frequency': 0.0, 'phase': 0.0}] #, 'frequency': 2.0 * 1/2/np.pi,'phase': np.pi/2}]


#for fwhm in FWHMs:
#    field_params += [{'amplitude': 0.1, 'FWHM': fwhm}]





### Making a job request -- looping temperatures

In [12]:
simulation_type = 'Undriven Thermal equilibrium'
simulation_desc = 'Equilibrium generation -- regular damping'

_loop_kwargs = 'system_parameters'
evolution_timesteps = 1000
initial_state_timestamp = '1782196253' #'1781702763' #'1780759388' #'1780908007' #'1780759388'
initial_state_index = 4
grid_parameters = {'time_sampling': 1501, 'time_duration': 2 * np.pi * 5}
vector_potential = np.ones(evolution_timesteps) * 0.0
#temp_list = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8]
temp_list = [0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5,0.55,0.6,0.65,0.7,0.75,0.8,0.85,0.9,0.95,1.0,1.05,1.1,1.3,1.5,2.0]
#temp_list = np.linspace(0.1,1.2,12,endpoint=True)
#temp_list = [0.5]
#for temp in temp_list:
#    system_parameters += []
system_parameters = []
#vector_potential = []
occupation_tracking_every_n = None

save_full_state = True

for temp in temp_list:
    system_parameters += [{'critical_temperature': 1, 'temperature': temp, 'eta': 0.2}]

field_type = None
field_params = None



### Creating jobs

In [21]:
crt_run_parameters = file_management.single_variable_kwargs_array(_loop_tag=_loop_kwargs, num_timesteps = evolution_timesteps, system_parameters = system_parameters, initial_state_timestamp = initial_state_timestamp, initial_state_index=initial_state_index, grid_parameters=grid_parameters, field_type=field_type, field_params=field_params, track_every_n=occupation_tracking_every_n, save_full_state = save_full_state)


request_creation.create_request_folder_any_machine(                                                                                  
      calculation_type = simulation_type,                                                                                              
      machine_identifier = crt_simulations_machine,                                                                                    
      kwargs_array = crt_run_parameters,                                                                                               
      job_memory_unit = 'G',                                                                                                           
      job_memory_value = 8,                                                                                                          
      job_time = (0,12,0,0),                                                                                                           
      task_summary = simulation_desc,                                                                                                  
      verbose = True,
      cpus_per_task = 1,
      # NEW v2-style parameters:
      solver_method_file = 'code_run.py',
      solver_method_name = 'evolve_keldysh_state',
      #cluster_backend = 'v2.2.0-beta'  # Or whichever version exists on cluster
  )

Created request folder: 1782293690


In [8]:
file_management.display_unattempted_folders(machine_identifier = crt_simulations_machine, folder_summaries = True)

Directory 1782291209 has not been attempted on cluster.

      Human-readable file created at Wed Jun 24 10:53:29 2026, using demler_tools v2.3.0-beta-17-g6b0081d
      Path convention: v3
      Created by user: fmarijanovic
      Project name: psBQP-keldysh
      
      Calculation type: Strong potential drive
      Number of jobs to run: 5
      
      Task summary: Strong Gaussian pulse response -- lower damping -- FWHM sweep
      
      Current folder state: unattempted
      
      Calculation log:





In [22]:
cluster_interface.push_unattempted_folders(machine_identifier = crt_simulations_machine)

KeyboardInterrupt: 

In [17]:
file_management.display_all_folders(machine_identifier = crt_simulations_machine, folder_summaries = True)

Directory with timestamp 1780988799 has status: submitted.

      Human-readable file created at Tue Jun  9 09:06:39 2026, using demler_tools v2.3.0-beta-17-g6b0081d
      Path convention: v3
      Created by user: fmarijanovic
      Project name: psBQP-keldysh
      
      Calculation type: Equilibration
      Number of jobs to run: 24
      
      Task summary: Computing the equilibrium state for future reference
      
      Current folder state: submitted
      
      Calculation log:
      [Tue Jun  9 09:06:55 2026] Updated directory state to the following: submitted
      [Tue Jun  9 09:06:56 2026] Submitted folder job to cluster with controller cluster_files/cluster_control_0_main.sh and under ID: 2660482
      [Tue Jun  9 09:06:56 2026] Submitted automatic postprocess job to cluster with controller cluster_files/cluster_control_0_post.sh and under ID: 2660484



Directory with timestamp 1780988780 has status: submitted.

      Human-readable file created at Tue Jun  9 09:06:20 